In [1]:
%pip install gradio_client

/bin/zsh: /home/prometheus/anaconda3/envs/llama-env/lib/libncursesw.so.6: no version information available (required by /bin/zsh)
Note: you may need to restart the kernel to use updated packages.


In [2]:
from tqdm import tqdm
from huggingface_hub import InferenceClient
import requests
import json
import os

# Params
VERBOSE = True
CHOSEN = 'qwen'
FILE_NAMES = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged','7-wonders']
IT = 5
BASE_URL = 'https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/'
OVERWRITE = ['']
try:
    #DRIVE
    from google.colab import drive
    drive.mount('/content/drive',force_remount=True)
    BASE_FOLDER = 'drive/MyDrive/NLP_proj/'
    N_GPU_LAYERS = -1
except:
    #LOCAL
    BASE_FOLDER = './'
    N_GPU_LAYERS = 20
RULE_FOLDER = os.path.join(BASE_FOLDER, 'rules/texts/')
PROMPT_FOLDER = os.path.join(BASE_FOLDER, 'prompts/')
OUT_FOLDER = os.path.join(BASE_FOLDER, 'extraction/')

In [4]:
with open(f'{BASE_FOLDER}/hf_api_key', 'r') as f:
    api_key = f.read()


client = InferenceClient(
    model="openai/gpt-oss-120b",
    api_key=api_key[:-1],
)

def generate_message(sys_prompt, usr_prompt):
    return [
        {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": usr_prompt,
            },
    ]

In [5]:
prompt_depth1 = requests.get(BASE_URL+'prompts/extraction_depth1').text[:-42]
prompt_other = requests.get(BASE_URL+'prompts/extraction_other').text

In [6]:
with open(BASE_FOLDER + 'prompts/extraction_depth1') as f:
    prompt_depth1 = f.read()
#print(prompt_depth1)
with open(BASE_FOLDER + 'prompts/extraction_other') as f:
    prompt_other = f.read()
#print(prompt_other.format(goal='hello'))

prompt_reflection = """
    Check and refine your results based on the following criteria :
(1) Are there any mechanics that are not **DIRECTLY** related to "{goal}"? 
(2) Are there any mechanics that are related to "{goal}" but depend from other already extracted?
If any mechanic contribute but not directly to the goal or depend on other extracted in this step remove it from your response
Respond a complete result in the same format as before and with nothing else.
"""
"""
EXAMPLES:
correct reflection: since drawing a card does not contribute directly to the scores, but is only one step to claim routes (the action that directly contribute to the score), should be removed
wrong reflection: dwawing cards directly contributes to the goal by providing the necessary resources to claim routes and complete tickets.

correct reflection: Increasing the number of action does not directly change the amount of shields, instead depends on other mechanics to do so, for this reason will be discarded
wrong reflection: Increasing the number of Actions available allows you to play more cards, including Action cards that can gain more <shield> or buy better cards."""
print(prompt_reflection.format(goal='GOAL'))


    Check and refine your results based on the following criteria :
(1) Are there any mechanics that are not **DIRECTLY** related to "GOAL"? 
(2) Are there any mechanics that are related to "GOAL" but depend from other already extracted?
If any mechanic contribute but not directly to the goal or depend on other extracted in this step remove it from your response
Respond a complete result in the same format as before and with nothing else.



In [7]:
to_do = [g for g in FILE_NAMES if f'{g}_remote.json' not in os.listdir(OUT_FOLDER)
                                 or f'{g}_remote.json' in OVERWRITE]

#out = client.chat_completion(messages=generate_message(sys_prompt, usr_prompt), temperature=0.7)
#out = out.choices[0].message.content

REFLECTION = False
if to_do == []:
    print('Nothing to do')
MAX_EXTRACTION_DEPTH = 4
for g in to_do:
    depth = {}
    rulebook = requests.get(BASE_URL +'rules/texts/'+g+'.txt').text
    name = g.replace('_',' ')
    messages = generate_message( prompt_depth1 , f"\nHere is the full rulebook of the game {name}:\n"+rulebook)
    out = client.chat_completion(messages=messages,temperature=0.7).choices[0].message
    messages.append({'role': out.role, 'content':out.content})
    out = out.content
    if(VERBOSE): print(f'pre-reflection: {out}')
    if(REFLECTION):
        messages.append({
            "role": "user",
            "content": prompt_reflection.format(goal='the end of the game')
        })
        reflection = client.chat_completion(messages=messages,temperature=0.7).choices[0].message
        messages.append({'role': reflection.role, 'content': reflection.content})
        out = reflection.content
        if(VERBOSE): print(f'after reflection: {out}')
    try:
        js = json.loads(f'{{"": {out} }}')
    except:
        out = f'{{"": { out[8:-4] } }}'
        js = json.loads(out)
        
    depth['1'] = [e['name'] for e in js['']]
    if(VERBOSE): print(depth['1'])
    for d in range(2,MAX_EXTRACTION_DEPTH):
        depth[str(d)] = []
        for e in depth[str(d-1)]:
            if(VERBOSE): print(f'descending from: {e}')
            messages.append({
                    "role": "user",
                    "content": prompt_other.format(goal=e)
                })
            #print(prompt_other.format(goal=e))
            out = client.chat_completion(messages=messages,temperature=0.7).choices[0].message
            messages.append({'role': out.role, 'content':out.content})
            out = out.content
            if(VERBOSE): print(f'pre-reflection: {out}')
            if(REFLECTION):
                messages.append({
                        "role": "user",
                        "content": prompt_reflection.format(goal=e)
                    })
                #print(prompt_reflection.format(goal=e))
                reflection = client.chat_completion(messages=messages,temperature=0.7).choices[0].message
                messages.append({'role': reflection.role, 'content': reflection.content})
                out = reflection.content
                if(VERBOSE): print(f'after reflection: {out}')
                
                
            
            try:
                js = json.loads(f'{{"": {out} }}')
            except:
                out = f'{{"": { out[8:-4] } }}'
                js = json.loads(out)
            depth[str(d)] += [e['name'] for e in js['']]
    with open(f'{OUT_FOLDER}/{g}_remote.json', 'w') as f:
        json.dump(depth, f, indent=4)

pre-reflection: [
    {
        "name": "Build houses (connect cities)",
        "type": "Contribute",
        "description": "Players spend Elektro to place houses on city spaces, increasing the number of cities in their network.",
        "reasoning": "The more cities a player connects, the higher the maximum number of cities they can potentially power in the final scoring, directly affecting the winning condition."
    },
    {
        "name": "Auction power plants",
        "type": "Contribute",
        "description": "Players bid for power‑plant cards; each plant’s number indicates how many cities it can supply when powered.",
        "reasoning": "Acquiring higher‑capacity power plants raises the total number of cities a player can power, which is the primary determinant of victory."
    },
    {
        "name": "Buy resources",
        "type": "Contribute",
        "description": "Players purchase coal, oil, garbage or uranium tokens from the resource market to fuel their power 

## Restructure previous output

In [11]:
with open(BASE_FOLDER + 'prompts/extraction_restruct') as f:
    prompt_restruct = f.read()

In [18]:
to_do = [g for g in FILE_NAMES if f'{g}_final.json' not in os.listdir(OUT_FOLDER)
                                 or f'{g}_final.json' in OVERWRITE]
for g in to_do:
    name = g.replace('_', ' ')
    print(name)
    # SYS PROMPT
    messages = [{'role': 'system', 'content': prompt_restruct}]
    out = client.chat_completion(messages=messages,temperature=0.7).choices[0].message
    messages.append({'role': out.role, 'content':out.content})
    out = out.content
    print(out)
    # RULEBOOK
    rulebook = requests.get(BASE_URL +'rules/texts/'+g+'.txt').text
    messages.append({'role': 'user', 'content': f"Here is the full rulebook for {name}, read it carefully and reply with ready when you are ready to receive the extraction text\n{'-'*10}{rulebook}"})
    out = client.chat_completion(messages=messages,temperature=0.7).choices[0].message
    messages.append({'role': out.role, 'content':out.content})
    out = out.content
    print(out)
    # HIERARCHY
    with open(BASE_FOLDER + f'extraction/{g}_remote.json') as f:
        hierarchy =  f.read()
    retry = 10
    while retry > 0:
        messages.append({'role': 'user', 'content': f"Here is the hierarchy for the game {name}.\n Now produce the corrected hierarchy JSON only\n{'-'*10}{hierarchy}"})
        out = client.chat_completion(messages=messages,temperature=0.7).choices[0].message.content
        try:
            out = json.loads(out)
            retry = 0
        except:
            print(retry)
            retry -=1
    print(out)
    with open(BASE_FOLDER + f'extraction/{g}_final.json', 'w') as f:
        json.dump(out,f, indent=4)

dominion
Please provide the rulebook text and the extracted hierarchy JSON so I can create the corrected hierarchy.
ready
{'0': ['Most victory points wins'], '1': ['Game ends if Province pile empty', 'Game ends if three supply piles empty', 'Victory points counted from all cards', 'Tie broken by fewer turns', 'Victory cards provide victory points', 'Curse cards provide negative victory points', 'Gardens give victory points per ten cards'], '2': ['Buy a Victory card', 'Gain a Victory card', 'Trash a card to gain a more expensive Victory card', 'Supply depletion blocks gaining Victory cards', 'Buy a Curse card', 'Gain a Curse card', 'Attack card gives Curse to other players', 'Moat reaction blocks Attack curses', 'Play Witch gives Curse', 'Insufficient coins or Buys prevents buying Victory cards', 'Play Treasure cards to generate coins', 'Play Action cards that give +Coin', 'Play Action cards that give +Buy', 'Play Action cards that give +Card', 'Start turn with 1 Buy token', 'Start turn

In [15]:
t = json.loads(out)

In [16]:
t

{'0': ['Player with most points wins'],
 '1': ['Earn points by claiming routes',
  'Earn points by completed tickets',
  'Lose points for uncompleted tickets',
  'Earn 10 bonus points for longest continuous path',
  'Tie‑break: most completed tickets then longest path'],
 '2': ['Claim a route action',
  'Draw tickets action',
  'Draw train cards action',
  'Must have matching train cards equal to route length',
  'Must have enough plastic trains for route length',
  'Route must be unclaimed (open)',
  'Cannot claim both routes of a double‑route in 2‑3 player games',
  'Only one route may be claimed per turn',
  'Must claim entire route in one turn',
  'Score route points using length table',
  'Ticket draw requires keeping at least one ticket',
  'Tickets kept secret until end of game',
  'Longest path measured by continuous same‑color trains',
  'All tied players receive longest‑path bonus',
  'Game ends when a player ends turn with 0‑2 trains remaining'],
 '3': ['Turn: choose exactly